# Tariikhna — Model Comparison Pipeline
**5 Models × 2 Prompts Each → Side-by-Side Comparison**

### Models tested:
| # | Model ID | Style |
|---|----------|-------|
| 1 | `fal-ai/flux/dev` | FLUX Dev — best quality |
| 2 | `fal-ai/flux-pro` | FLUX Pro — photorealistic |
| 3 | `fal-ai/flux-pro/v1.1` | FLUX Pro v1.1 — latest |
| 4 | `fal-ai/stable-diffusion-v3-medium` | SD3 Medium — artistic |
| 5 | `fal-ai/ideogram/v2` | Ideogram v2 — illustration |

### Prompt strategies:
- **Prompt A** — Silhouette / no-face style (Islamic compliant, abstract characters)
- **Prompt B** — Watercolor illustration style (soft, children's book feel)

> ⚠️ **Removed**: `flux/schnell` (low quality), `Recraft v3` (bad results), `SD 3.5` (eliminated)

## Step 1 — Install dependencies

In [ ]:
!pip install requests Pillow fal-client groq --quiet

## Step 2 — Configuration
Set your API keys below.

In [ ]:
import os

# ── API KEYS ──────────────────────────────────────────────────
FAL_API_KEY  = "YOUR_FAL_KEY_HERE"   # https://fal.ai/dashboard/keys
GROQ_API_KEY = "YOUR_GROQ_KEY_HERE"  # https://console.groq.com/keys
# ─────────────────────────────────────────────────────────────

os.environ["FAL_KEY"] = FAL_API_KEY

# ── 5 IMAGE MODELS TO COMPARE ────────────────────────────────
IMAGE_MODELS = {
    "FLUX Dev":        "fal-ai/flux/dev",
    "FLUX Pro":        "fal-ai/flux-pro",
    "FLUX Pro v1.1":   "fal-ai/flux-pro/v1.1",
    "SD3 Medium":      "fal-ai/stable-diffusion-v3-medium",
    "Ideogram v2":     "fal-ai/ideogram/v2",
}

# REMOVED (bad results): flux/schnell, Recraft v3 I, SD 3.5

print("✓ Config set")
print(f"  LLM   : llama-3.1-8b via Groq")
print(f"  Models: {len(IMAGE_MODELS)} image models")
for name, model_id in IMAGE_MODELS.items():
    print(f"    • {name}: {model_id}")

## Step 3 — Load passage
We use one passage for the full comparison — change `PASSAGE_INDEX` to test others.

In [ ]:
import json

# ── Load passages ─────────────────────────────────────────────
with open("curated_passages_287.json", "r", encoding="utf-8") as f:
    passages = json.load(f)

# ── CHOOSE A PASSAGE ──────────────────────────────────────────
PASSAGE_INDEX = 0   # 0 to 286
# ─────────────────────────────────────────────────────────────

passage = passages[PASSAGE_INDEX]

print(f"Loaded {len(passages)} passages")
print(f"Testing  : {passage['id']}")
print(f"Chapter  : {passage['chapter_title']}")
print(f"Types    : {passage.get('scene_types', [])}")
print()
print("PASSAGE TEXT:")
print("-" * 60)
print(passage["passage"])
print("-" * 60)

## Step 4 — Generate structured panel via LLM (Groq)
Parse the passage into a structured scene description. This runs once — both prompt variants share the same panel metadata.

In [ ]:
import time
from groq import Groq

SYSTEM_PROMPT = """You are Tariikhna, an AI that converts Islamic historical narratives into structured comic panel descriptions for children aged 6-12.

Given a passage, output ONE JSON object describing ONE comic panel.

CRITICAL ISLAMIC DEPICTION RULES:
- Prophets (Muhammad, Ibrahim, Musa, Isa, Ismail, ALL prophets): NEVER show face.
  depiction_rule must be SILHOUETTE (preferred), FROM_BEHIND, or NO_FACE.
- Angels: represent as glowing light or golden aura only, NEVER humanoid form.
- Sahaba (Abu Bakr, Umar, Ali, etc.): faces allowed, depiction_rule = FULL.
- Women: modest clothing (hijab/khimar covering hair). Face and hands may be visible.
- No graphic violence. Battles shown through dust, distant silhouettes only.
- No modern objects. Historically accurate 7th century Arabia.
- Children aged 6-12 audience — uplifting, non-frightening.

ALL characters should lean toward silhouette / shadow style — minimize detailed facial features even for FULL characters. Think children's Islamic picture book aesthetic.

Output ONLY valid JSON:
{
  "scene_title": "short title",
  "era": "one of: ancient_prophets | pre_prophetic | early_makkah | late_makkah | madinah_early | madinah_late",
  "region": "one of: makkah | madinah | arabian_desert | egypt_ancient | levant_ancient | sinai | abyssinia | other",
  "source_reference": "source text",
  "characters": [
    {
      "id": "lowercase_id",
      "name": "Full Name",
      "role": "prophet | sahabi | family_of_prophet | antagonist | supporting | crowd | angel",
      "depiction_rule": "SILHOUETTE | FROM_BEHIND | NO_FACE | FULL",
      "appearance": "clothing colors, height, build — no face details"
    }
  ],
  "moral_lesson": "Islamic value this scene teaches",
  "narrative_text": "simple engaging story for children aged 6-12, 2-3 sentences",
  "compliance": {
    "prophet_check": "NO_PROPHET_IN_SCENE | PROPHET_SILHOUETTE | PROPHET_FROM_BEHIND | PROPHET_NOT_VISIBLE",
    "modesty_check": "COMPLIANT",
    "child_safe": "APPROPRIATE",
    "notes": "brief explanation"
  },
  "scene_description": "70-100 word visual scene description: characters, setting, mood, lighting. No style instructions — just what is in the scene."
}
Respond ONLY with valid JSON. No markdown. No explanation."""

FEW_SHOT = """EXAMPLE INPUT: "Abu Bakr wept with joy when the Prophet told him he would be his companion on the journey."
Source: Ibn Hisham, Al-Sira, Vol. 2

EXAMPLE OUTPUT:
{"scene_title":"Abu Bakr Learns He Will Accompany the Prophet","era":"late_makkah","region":"makkah","source_reference":"Ibn Hisham, Al-Sira Al-Nabawiyya, Vol. 2","characters":[{"id":"prophet_muhammad","name":"Prophet Muhammad (PBUH)","role":"prophet","depiction_rule":"SILHOUETTE","appearance":"Tall figure in white thobe and cloak, white turban, seen as a warm golden silhouette. No face visible."},{"id":"abu_bakr","name":"Abu Bakr al-Siddiq","role":"sahabi","depiction_rule":"FULL","appearance":"Slender older man, light olive skin, grey-streaked beard. Off-white thobe, brown woolen cloak, white head cloth."}],"moral_lesson":"True friendship means being ready to sacrifice everything for Allah's sake.","narrative_text":"When the Prophet told Abu Bakr he would travel with him, Abu Bakr cried happy tears. He had been waiting and preparing for this special day for so long!","compliance":{"prophet_check":"PROPHET_SILHOUETTE","modesty_check":"COMPLIANT","child_safe":"APPROPRIATE","notes":"Prophet shown as warm silhouette. Abu Bakr with gentle features, no graphic content."},"scene_description":"Interior of a modest mud-brick home in 7th century Makkah. A tall glowing silhouette of a man in white robe stands near an open doorway with warm golden afternoon light streaming behind him. A slender older man sits on a woven mat, tears of joy on his face, hand over his heart. Simple clay water jug in corner, small oil lamp. Warm, peaceful atmosphere."}
"""

user_message = f"""{FEW_SHOT}
Now convert this passage:

INPUT: "{passage['passage']}"
Source: {passage.get('source', passage.get('source_reference', ''))}

OUTPUT:"""

client = Groq(api_key=GROQ_API_KEY)
print("⏳ Calling Llama-3.1-8b via Groq...")
t0 = time.time()

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_message}
    ],
    temperature=0.7,
    max_tokens=2000,
    response_format={"type": "json_object"}
)

raw_response = response.choices[0].message.content
print(f"✓ Done in {time.time()-t0:.1f}s")
print()
print("RAW OUTPUT:")
print("-" * 60)
print(raw_response)
print("-" * 60)

## Step 5 — Parse & validate panel

In [ ]:
import re

def parse_llm_output(raw_text):
    text = raw_text.strip()
    if "```" in text:
        match = re.search(r"```(?:json)?\s*([\s\S]+?)```", text)
        if match:
            text = match.group(1).strip()
    start = text.find("{")
    end   = text.rfind("}") + 1
    if start == -1 or end == 0:
        raise ValueError("No JSON found")
    return json.loads(text[start:end])

def validate_panel(panel):
    issues, warnings = [], []
    for field in ["scene_title", "era", "region", "characters", "moral_lesson",
                  "narrative_text", "compliance", "scene_description"]:
        if field not in panel:
            issues.append(f"Missing field: {field}")
    for c in panel.get("characters", []):
        if c.get("role") == "prophet" and c.get("depiction_rule") == "FULL":
            issues.append(f"CRITICAL: Prophet '{c.get('name')}' shown FULL — must be SILHOUETTE/FROM_BEHIND")
    return issues, warnings

try:
    panel = parse_llm_output(raw_response)
    print("✓ JSON parsed")
except Exception as e:
    print(f"✗ Parse failed: {e}")
    raise

issues, warnings = validate_panel(panel)
if issues:
    for i in issues: print(f"✗ ISSUE: {i}")
else:
    print("✓ Compliance OK")

print()
print(f"Scene title : {panel.get('scene_title')}")
print(f"Era/Region  : {panel.get('era')} / {panel.get('region')}")
print(f"Characters  : {[c['name'] for c in panel.get('characters', [])]}")
print(f"Prophet     : {panel.get('compliance', {}).get('prophet_check')}")
print()
print("Scene description:")
print(" ", panel.get('scene_description', ''))

## Step 6 — Build the 2 prompt variants

Both prompts describe the **same scene** from `scene_description`, but with different artistic styles:
- **Prompt A** — Silhouette / Islamic shadow art style (no facial features, high contrast)
- **Prompt B** — Soft watercolor / children's book illustration style

In [ ]:
scene = panel.get("scene_description", "")
chars = panel.get("characters", [])

# Build character constraint string
char_rules = []
for c in chars:
    rule = c.get("depiction_rule", "FULL")
    name = c.get("name", "Character")
    if rule in ["SILHOUETTE", "NO_FACE", "FROM_BEHIND"]:
        char_rules.append(f"{name}: shown as silhouette, no facial features visible")
    else:
        char_rules.append(f"{name}: visible but minimal facial detail, soft features")

char_constraint = ". ".join(char_rules)

# ── PROMPT A: Silhouette / Shadow Art ───────────────────────
PROMPT_A = f"""Islamic children's picture book illustration, silhouette art style. \
Warm golden and amber color palette, dramatic backlit lighting. \
ALL human figures shown as elegant silhouettes with NO facial features — \
figures defined only by clothing shapes and posture. \
{char_constraint}. \
Scene: {scene} \
Flat geometric shapes, clean lines, high contrast between warm light and shadow. \
7th century Arabian setting, mud-brick architecture, desert landscape. \
No text. No modern objects. No facial features on any person."""

# ── PROMPT B: Watercolor Children's Book ────────────────────
PROMPT_B = f"""Soft watercolor children's book illustration. \
Gentle muted palette, warm earth tones, soft diffused lighting. \
Characters rendered with minimal facial detail — expressive body language, \
loose brushstroke faces with no sharp features. \
{char_constraint}. \
Scene: {scene} \
Painterly style, visible watercolor texture, gentle washes of color. \
Historically accurate 7th century Arabia — mud-brick buildings, woven mats, clay pots. \
Uplifting, peaceful mood suitable for children aged 6-12. \
No text. No modern objects."""

PROMPTS = {
    "A — Silhouette": PROMPT_A,
    "B — Watercolor": PROMPT_B,
}

print("✓ Prompts built")
print()
print(f"PROMPT A ({len(PROMPT_A.split())} words):")
print("-" * 60)
print(PROMPT_A)
print()
print(f"PROMPT B ({len(PROMPT_B.split())} words):")
print("-" * 60)
print(PROMPT_B)

## Step 7 — Generate images (5 models × 2 prompts = 10 images)
This cell runs all 10 generations. Estimated time: 3-8 minutes depending on queue.

In [ ]:
import fal_client
import time

# ── Model-specific argument builders ─────────────────────────
def build_args(model_id, prompt):
    """Return the right arguments dict for each model."""

    # FLUX Dev
    if model_id == "fal-ai/flux/dev":
        return {
            "prompt": prompt,
            "image_size": "landscape_4_3",
            "num_inference_steps": 28,
            "guidance_scale": 3.5,
            "num_images": 1,
            "enable_safety_checker": True,
        }

    # FLUX Pro (original)
    elif model_id == "fal-ai/flux-pro":
        return {
            "prompt": prompt,
            "image_size": "landscape_4_3",
            "num_inference_steps": 28,
            "guidance_scale": 3.5,
            "num_images": 1,
            "safety_tolerance": "2",
        }

    # FLUX Pro v1.1
    elif model_id == "fal-ai/flux-pro/v1.1":
        return {
            "prompt": prompt,
            "image_size": "landscape_4_3",
            "num_images": 1,
            "safety_tolerance": "2",
        }

    # SD3 Medium
    elif model_id == "fal-ai/stable-diffusion-v3-medium":
        return {
            "prompt": prompt,
            "image_size": "landscape_4_3",
            "num_inference_steps": 28,
            "guidance_scale": 7.0,
            "num_images": 1,
            "enable_safety_checker": True,
        }

    # Ideogram v2
    elif model_id == "fal-ai/ideogram/v2":
        return {
            "prompt": prompt,
            "aspect_ratio": "4:3",
            "style_type": "ILLUSTRATION",
            "magic_prompt_option": "OFF",
        }

    # Fallback
    else:
        return {
            "prompt": prompt,
            "image_size": "landscape_4_3",
            "num_images": 1,
        }

# ── Run all generations ───────────────────────────────────────
results = {}  # {model_name: {prompt_label: url}}
errors  = {}

total = len(IMAGE_MODELS) * len(PROMPTS)
count = 0

print(f"⏳ Generating {total} images ({len(IMAGE_MODELS)} models × {len(PROMPTS)} prompts)...")
print()

for model_name, model_id in IMAGE_MODELS.items():
    results[model_name] = {}
    for prompt_label, prompt_text in PROMPTS.items():
        count += 1
        print(f"[{count}/{total}] {model_name} | {prompt_label}")
        t0 = time.time()
        try:
            args = build_args(model_id, prompt_text)
            res  = fal_client.run(model_id, arguments=args)

            # Extract URL (different models use different response keys)
            if "images" in res and res["images"]:
                url = res["images"][0]["url"]
            elif "image" in res:
                url = res["image"]["url"]
            else:
                raise ValueError(f"No image URL in response: {list(res.keys())}")

            results[model_name][prompt_label] = url
            print(f"  ✓ {time.time()-t0:.1f}s — {url[:60]}...")

        except Exception as e:
            errors[f"{model_name}/{prompt_label}"] = str(e)
            print(f"  ✗ Error: {e}")

        time.sleep(1)  # polite delay

print()
print(f"✓ Done. {total - len(errors)} succeeded, {len(errors)} failed.")
if errors:
    print("Errors:")
    for k, v in errors.items():
        print(f"  {k}: {v}")

## Step 8 — Download images & add context overlay
Each image gets a text overlay with: scene title, narrative text, and moral lesson.

In [ ]:
import requests
from PIL import Image, ImageDraw, ImageFont
from io import BytesIO
import textwrap
import os

os.makedirs("comparison_output", exist_ok=True)

# ── Font loader ───────────────────────────────────────────────
def get_fonts():
    font_paths = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
        "/Library/Fonts/Arial Bold.ttf",
        "C:/Windows/Fonts/arialbd.ttf",
    ]
    font_paths_regular = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
        "/Library/Fonts/Arial.ttf",
        "C:/Windows/Fonts/arial.ttf",
    ]
    bold = regular = None
    for p in font_paths:
        if os.path.exists(p):
            bold = p; break
    for p in font_paths_regular:
        if os.path.exists(p):
            regular = p; break
    try:
        f_title  = ImageFont.truetype(bold or regular, 18) if (bold or regular) else ImageFont.load_default()
        f_body   = ImageFont.truetype(regular or bold, 13) if (bold or regular) else ImageFont.load_default()
        f_small  = ImageFont.truetype(regular or bold, 11) if (bold or regular) else ImageFont.load_default()
    except:
        f_title = f_body = f_small = ImageFont.load_default()
    return f_title, f_body, f_small

font_title, font_body, font_small = get_fonts()

# ── Panel renderer with context overlay ───────────────────────
def render_panel_with_context(img_url, panel_data, model_name, prompt_label):
    """Download image and add title bar + narrative overlay + moral lesson bar."""

    # Download
    r = requests.get(img_url, timeout=30)
    img = Image.open(BytesIO(r.content)).convert("RGB")

    # Resize to standard width
    TARGET_W = 900
    ratio = TARGET_W / img.width
    img = img.resize((TARGET_W, int(img.height * ratio)), Image.LANCZOS)

    img_w, img_h = img.size
    pad = 16

    # Layout areas
    header_h  = 48   # model + prompt label
    title_h   = 42   # scene title
    footer_h  = 70   # narrative + moral
    total_h   = header_h + title_h + img_h + footer_h

    canvas = Image.new("RGB", (img_w, total_h), (250, 246, 238))
    draw   = ImageDraw.Draw(canvas)

    # ── Header bar (model name + prompt label) ────────────────
    draw.rectangle([0, 0, img_w, header_h], fill=(30, 60, 45))
    draw.text((pad, 8),  model_name,   fill=(200, 230, 210), font=font_title)
    draw.text((pad, 26), prompt_label, fill=(160, 200, 175), font=font_small)

    # ── Scene title bar ───────────────────────────────────────
    draw.rectangle([0, header_h, img_w, header_h + title_h], fill=(55, 100, 70))
    scene_title = panel_data.get("scene_title", "")
    draw.text((pad, header_h + 12), scene_title, fill=(255, 248, 220), font=font_title)

    # Compliance badge (top-right of title bar)
    comp_check = panel_data.get("compliance", {}).get("prophet_check", "")
    badge_text = comp_check.replace("PROPHET_", "").replace("NO_PROPHET_IN_SCENE", "NO PROPHET")
    badge_col  = (50, 130, 80) if comp_check in ["NO_PROPHET_IN_SCENE", "PROPHET_SILHOUETTE",
                  "PROPHET_FROM_BEHIND", "PROPHET_NOT_VISIBLE"] else (180, 60, 60)
    bx = img_w - 160
    draw.rectangle([bx, header_h + 6, img_w - 6, header_h + title_h - 6], fill=badge_col)
    draw.text((bx + 6, header_h + 14), badge_text, fill=(255, 255, 255), font=font_small)

    # ── Comic image ────────────────────────────────────────────
    canvas.paste(img, (0, header_h + title_h))

    # ── Footer: narrative + moral ──────────────────────────────
    footer_y = header_h + title_h + img_h
    draw.rectangle([0, footer_y, img_w, total_h], fill=(245, 240, 225))

    # Narrative text
    narrative = panel_data.get("narrative_text", "")
    nav_wrap  = textwrap.fill(narrative, width=100)
    draw.multiline_text((pad, footer_y + 6), nav_wrap,
                        fill=(40, 30, 20), font=font_body, spacing=4)

    # Moral lesson (bottom strip)
    moral_y = total_h - 28
    draw.rectangle([0, moral_y - 4, img_w, total_h], fill=(220, 200, 160))
    moral = "✦  " + panel_data.get("moral_lesson", "")
    draw.text((pad, moral_y), textwrap.fill(moral, width=110),
              fill=(70, 45, 15), font=font_small)

    return canvas

# ── Process all results ───────────────────────────────────────
rendered = {}  # {model_name: {prompt_label: PIL.Image}}

print("⏳ Downloading and rendering panels...")
print()

for model_name, prompt_dict in results.items():
    rendered[model_name] = {}
    for prompt_label, url in prompt_dict.items():
        print(f"  {model_name} | {prompt_label}")
        try:
            img = render_panel_with_context(url, panel, model_name, prompt_label)
            rendered[model_name][prompt_label] = img

            # Save individual panel
            safe_model  = model_name.replace(" ", "_").replace("/", "-")
            safe_prompt = prompt_label.split(" ")[0]  # "A" or "B"
            save_path   = f"comparison_output/{safe_model}_Prompt{safe_prompt}.png"
            img.save(save_path)
            print(f"    ✓ Saved: {save_path}")

        except Exception as e:
            print(f"    ✗ Error: {e}")

print()
print("✓ All panels rendered")

## Step 9 — Build final comparison grid
Creates a single large image with all 5 models × 2 prompts laid out in a grid.
Rows = models, Columns = Prompt A / Prompt B

In [ ]:
from IPython.display import display

def build_comparison_grid(rendered_dict, panel_data, passage_data):
    """Assemble a full comparison grid: rows=models, cols=prompts."""

    prompt_labels = list(PROMPTS.keys())
    model_names   = list(rendered_dict.keys())

    # Get a sample image to determine cell size
    sample = None
    for m in model_names:
        for p in prompt_labels:
            if p in rendered_dict.get(m, {}):
                sample = rendered_dict[m][p]; break
        if sample: break

    if sample is None:
        print("✗ No rendered images found")
        return None

    cell_w, cell_h = sample.size
    cols       = len(prompt_labels)
    rows       = len(model_names)
    header_h   = 80   # top header for column labels
    gap        = 8
    grid_w     = cols * cell_w + (cols + 1) * gap
    grid_h     = header_h + rows * (cell_h + gap) + gap

    grid  = Image.new("RGB", (grid_w, grid_h), (200, 195, 185))
    draw  = ImageDraw.Draw(grid)

    # Top header
    draw.rectangle([0, 0, grid_w, header_h], fill=(20, 45, 30))
    title = f"Tariikhna — Model Comparison | {panel_data.get('scene_title', '')} | {passage_data.get('id', '')}"
    draw.text((gap, 12), title, fill=(220, 245, 220), font=font_title)

    # Column labels
    for ci, pl in enumerate(prompt_labels):
        cx = gap + ci * (cell_w + gap)
        draw.text((cx + 10, 40), pl, fill=(180, 220, 190), font=font_body)

    # Place images
    for ri, model_name in enumerate(model_names):
        y = header_h + ri * (cell_h + gap) + gap
        for ci, pl in enumerate(prompt_labels):
            x = gap + ci * (cell_w + gap)
            img = rendered_dict.get(model_name, {}).get(pl)
            if img:
                grid.paste(img, (x, y))
            else:
                # Draw placeholder
                draw.rectangle([x, y, x + cell_w, y + cell_h], fill=(160, 155, 145))
                draw.text((x + 20, y + cell_h // 2), f"FAILED: {model_name}",
                          fill=(255, 255, 255), font=font_body)

    return grid


print("⏳ Building comparison grid...")
grid = build_comparison_grid(rendered, panel, passage)

if grid:
    scene_safe = panel.get("scene_title", "comparison").replace(" ", "_")[:40]
    grid_path  = f"comparison_output/COMPARISON_{scene_safe}.png"
    grid.save(grid_path, quality=95)
    print(f"✓ Comparison grid saved: {grid_path}")
    print(f"  Size: {grid.width} × {grid.height} px")
    print()

    # Display (may be large — resize for notebook display)
    display_w = 1200
    ratio      = display_w / grid.width
    grid_small = grid.resize((display_w, int(grid.height * ratio)), Image.LANCZOS)
    display(grid_small)

## Step 10 — Per-model summary report
Quick visual inspection of each model's output with a score card.

In [ ]:
from IPython.display import display, HTML

# Print HTML summary table
rows_html = ""
for model_name in IMAGE_MODELS.keys():
    for prompt_label in PROMPTS.keys():
        url = results.get(model_name, {}).get(prompt_label, None)
        status = "✓" if url else "✗"
        safe_model  = model_name.replace(" ", "_").replace("/", "-")
        safe_prompt = prompt_label.split(" ")[0]
        file_path   = f"comparison_output/{safe_model}_Prompt{safe_prompt}.png"
        rows_html += f"""
        <tr>
            <td><b>{model_name}</b></td>
            <td>{prompt_label}</td>
            <td>{status}</td>
            <td style='font-size:11px'>{url[:60] + '...' if url else 'FAILED'}</td>
            <td>{file_path if url else '-'}</td>
        </tr>"""

html = f"""
<h2>Tariikhna Model Comparison — Summary</h2>
<h3>Scene: {panel.get('scene_title', '')}</h3>
<p>Passage: {passage.get('id', '')} | Chapter: {passage.get('chapter_title', '')}</p>
<table border='1' cellpadding='6' style='border-collapse:collapse; font-family:monospace'>
  <thead>
    <tr style='background:#2d3e30; color:white'>
      <th>Model</th><th>Prompt</th><th>Status</th><th>URL</th><th>Saved file</th>
    </tr>
  </thead>
  <tbody>{rows_html}</tbody>
</table>
<br>
<b>Comparison grid:</b> comparison_output/COMPARISON_{panel.get('scene_title','').replace(' ','_')[:40]}.png
"""
display(HTML(html))

## Step 11 — Display individual results per model
Run this to inspect each model's Prompt A vs Prompt B side by side.

In [ ]:
from IPython.display import display

for model_name in IMAGE_MODELS.keys():
    print(f"\n{'='*60}")
    print(f"  MODEL: {model_name}")
    print(f"{'='*60}")

    imgs = []
    for prompt_label in PROMPTS.keys():
        img = rendered.get(model_name, {}).get(prompt_label)
        if img:
            imgs.append(img)
            print(f"  {prompt_label}: ✓")
        else:
            print(f"  {prompt_label}: ✗ FAILED")

    if imgs:
        # Combine A and B side by side
        gap = 8
        max_h = max(i.height for i in imgs)
        combined_w = sum(i.width for i in imgs) + gap * (len(imgs) + 1)
        combined   = Image.new("RGB", (combined_w, max_h + gap * 2), (180, 175, 165))
        x = gap
        for i in imgs:
            combined.paste(i, (x, gap))
            x += i.width + gap

        # Scale down for display
        dw = min(1400, combined_w)
        dr = dw / combined_w
        combined_small = combined.resize((dw, int(combined.height * dr)), Image.LANCZOS)
        display(combined_small)

## Step 12 — Save full JSON report

In [ ]:
report = {
    "passage_id"    : passage.get("id"),
    "chapter_title" : passage.get("chapter_title"),
    "input_passage" : passage.get("passage"),
    "panel"         : panel,
    "prompts": {
        "A_silhouette": PROMPT_A,
        "B_watercolor" : PROMPT_B,
    },
    "results": results,
    "errors" : errors,
    "models" : IMAGE_MODELS,
}

report_path = f"comparison_output/report_{passage.get('id', 'unknown')}.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(f"✓ Full report saved: {report_path}")
print()
print("── FINAL SUMMARY ──────────────────────────────────────")
print(f"  Passage      : {passage.get('id')} — {passage.get('chapter_title')}")
print(f"  Scene title  : {panel.get('scene_title')}")
print(f"  Models tested: {len(IMAGE_MODELS)}")
print(f"  Prompts/model: {len(PROMPTS)}")
print(f"  Total images : {len(IMAGE_MODELS) * len(PROMPTS)}")
print(f"  Succeeded    : {len(IMAGE_MODELS) * len(PROMPTS) - len(errors)}")
print(f"  Failed       : {len(errors)}")
print(f"  Output folder: comparison_output/")
print("────────────────────────────────────────────────────────")